[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Validation and table=True &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `CheckedHero` and `HeroDraft` classes its worked
examples wrote. Run it first. The tasks follow one another, and the last cell removes the scratch
folder.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from pydantic import ValidationError, field_validator
from sqlalchemy import event, insert, text
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes loaded")


class CheckedHero(SQLModel, table=True):
    """The same fields as Hero, with a rule about names, and a table."""

    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    age: int | None = None

    @field_validator("name")
    @classmethod
    def starts_with_a_capital(cls, value):
        print("      (the rule ran)")
        if not value[:1].isupper():
            raise ValueError("a hero's name starts with a capital letter")
        return value


class HeroDraft(SQLModel):
    """The same fields and the same rule, with no table."""

    name: str = Field(max_length=50)
    age: int | None = None

    @field_validator("name")
    @classmethod
    def starts_with_a_capital(cls, value):
        print("      (the rule ran)")
        if not value[:1].isupper():
            raise ValueError("a hero's name starts with a capital letter")
        return value

SQLModel.metadata.create_all(engine)


sqlmodel 0.0.42 | 8 heroes loaded


**1.** The two doors into a table model.


In [2]:
loose = CheckedHero(name="Spider-Boy", age="twelve")
print("the constructor:", repr(loose.age), type(loose.age).__name__)

try:
    CheckedHero.model_validate({"name": "Spider-Boy", "age": "twelve"})
except ValidationError as error:
    print("model_validate :", message(error).splitlines()[-1].strip()[:70])


the constructor: 'twelve' str
      (the rule ran)
model_validate : Input should be a valid integer, unable to parse string as an integer 


The constructor kept the word; `model_validate` named the field and what it expected.


**2.** A draft team, with a rule of its own.


In [3]:
class TeamDraft(SQLModel):
    name: str = Field(max_length=50)
    headquarters: str = Field(max_length=60)

    @field_validator("headquarters")
    @classmethod
    def long_enough(cls, value):
        if len(value) < 3:
            raise ValueError("a headquarters needs a real name")
        return value


try:
    TeamDraft(name="Z-Force", headquarters="Z")
except ValidationError as error:
    print("refused :", message(error).splitlines()[-1].strip()[:70])
print("accepted:", TeamDraft(name="Z-Force", headquarters="Sister Margaret's Bar").name)


refused : Value error, a headquarters needs a real name [type=value_error, input
accepted: Z-Force


The rule runs in the constructor, because this class has no table.


**3.** A batch, sorted into what passed and what did not.


In [4]:
def sorted_out(rows):
    """The drafts that passed, and a line for each row that did not."""
    passed, refused = [], []
    for number, row in enumerate(rows, start=1):
        try:
            passed.append(HeroDraft.model_validate(row))
        except ValidationError as error:
            refused.append(f"row {number}: {message(error).splitlines()[-1].strip()[:52]}")
    return passed, refused


passed, refused = sorted_out([{"name": "Black Lion", "age": "35"}, {"name": "tarantula", "age": 32},
                              {"name": "Dr. Weird", "age": "old"}])
print("passed :", [draft.name for draft in passed])
for line in refused:
    print("refused:", line)


      (the rule ran)
      (the rule ran)
      (the rule ran)
passed : ['Black Lion']
refused: row 2: Value error, a hero's name starts with a capital let
refused: row 3: Input should be a valid integer, unable to parse str


Two refusals with different reasons: one broke the rule about capital letters, and one could not be
made into a number.


**4.** A table model that checks what it is given.


In [5]:
class StrictTeam(SQLModel, table=True):
    model_config = {"validate_assignment": True}

    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    founded: int | None = None


try:
    StrictTeam(name="Preventers", founded="not a year")
except ValidationError as error:
    print("the constructor:", message(error).splitlines()[-1].strip()[:60])

team = StrictTeam(name="Preventers", founded="1994")                # converted on the way in
print("converted      :", repr(team.founded), type(team.founded).__name__)
try:
    team.founded = "still not a year"
except ValidationError as error:
    print("an assignment  :", message(error).splitlines()[-1].strip()[:60])


the constructor: Input should be a valid integer, unable to parse string as a
converted      : 1994 int
an assignment  : Input should be a valid integer, unable to parse string as a


`validate_assignment` closes the constructor as well, because SQLModel builds an object by assigning
each value. What it does not touch is a row read from the table.


**5.** A name too long for its column.


In [6]:
with engine.begin() as connection:
    connection.execute(text("INSERT INTO team (name, headquarters) VALUES (:name, 'Nowhere')"),
                       {"name": "T" * 80})

with Session(engine) as session:
    longest = max(session.exec(select(Team)).all(), key=lambda team: len(team.name))
    print("stored   :", len(longest.name), "characters")
    print("declared :", Team.__table__.columns["name"].type.length)


stored   : 80 characters
declared : 50


SQLite kept all eighty. The length is a promise the model makes and a rule only another database
would enforce.


**6.** A team, as a response would carry it.


In [7]:
class TeamSummary(SQLModel):
    id: int
    name: str


with Session(engine) as session:
    preventers = session.get(Team, 1)
    print(fields(TeamSummary.model_validate(preventers)))


{'id': 1, 'name': 'Preventers'}


`model_validate` read the fields off the object, since a model with no table takes its values from
attributes as readily as from a dictionary, and the headquarters is not in the summary at all.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Validation and table=True](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/06-validation-and-table-true.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
